## Question 2: Potato Disease Classification using CNN - SOLUTION

You are provided with the **Potato Disease Dataset** from PlantVillage, which contains images of potato leaves classified into three categories:

- **Early_blight**: Leaves affected by early blight disease
- **Late_blight**: Leaves affected by late blight disease  
- **healthy**: Healthy potato leaves

Your objective is to build a **Convolutional Neural Network (CNN) from scratch** using PyTorch to classify potato leaves into these 3 disease categories.
Your work will be evaluated based on the completion of the following tasks:

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

# Part 1: Load and Prepare Data

**Tasks:**

- Create a custom Dataset class or use ImageFolder (if applicable) 
- Create the training and testing datasets, and their DataLoaders
- Use RandomRotation(15) Augmentation on the training dataset 
- Display some sample images with their labels

In [ ]:
import os
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Define paths
train_dir = os.path.join(path, "PlantVillage", "train")
test_dir = os.path.join(path, "PlantVillage", "test")

# Define transforms
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

# Load datasets using ImageFolder
train_dataset = ImageFolder(root=train_dir, transform=train_transform)
test_dataset = ImageFolder(root=test_dir, transform=test_transform)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get a batch of training data
images, labels = next(iter(train_loader))

# Show images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = images[i]
    img = np.transpose(img.numpy(), (1, 2, 0))  # Convert (C, H, W) to (H, W, C)
    ax.imshow(img)
    ax.set_title(train_dataset.classes[labels[i].item()])
    ax.axis("off")

plt.tight_layout()
plt.show()

# Part 2: Build the CNN Model

**Tasks:**

Build a CNN from scratch with the following specifications:

- Create a CNN model class with **5 convolutional layers**
- Bonus: Add **BatchNormalization** after each convolutional layer

In [ ]:
import torch.nn as nn

class CNNModel(nn.Module):
    def __init__(self, num_classes=3):
        super(CNNModel, self).__init__()
        
        # Conv Layer 1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        # Conv Layer 2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # Conv Layer 3
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Conv Layer 4
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        # Conv Layer 5
        self.conv5 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)
        
        # Activation and Pooling
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully Connected Layers
        self.fc1 = nn.Linear(512 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
    
    def forward(self, x):
        # Layer 1: 128x128 -> 64x64
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        
        # Layer 2: 64x64 -> 32x32
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        
        # Layer 3: 32x32 -> 16x16
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        
        # Layer 4: 16x16 -> 8x8
        x = self.pool(self.relu(self.bn4(self.conv4(x))))
        
        # Layer 5: 8x8 -> 4x4
        x = self.pool(self.relu(self.bn5(self.conv5(x))))
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Fully Connected
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        
        return x

# Test model
model = CNNModel(num_classes=3)
print(model)

# Part 3: Training and Validation Functions

**Tasks:**

- Define the training loop function
- Define the validation loop function

In [ ]:
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = outputs.max(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

# Part 4: Training

**Tasks:**

- Set up device, model, loss function, and optimizer
- Train the model 
- Plot the training and validation losses and training and validation accuracy

In [ ]:
import torch.optim as optim

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNModel(num_classes=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.2f}%, Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%")

In [ ]:
# Plot Loss
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Train Acc')
plt.plot(val_accuracies, label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training and Validation Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

# Part 5: Bonus - Residual Connection

**Task:**

Now let's see if we can improve our model with a residual connection!

1. **Redefine the CNN model** but add a **residual (skip) connection** from the **2nd convolutional layer** to the **4th convolutional layer**
   - You can use either **summation** OR **concatenation** for the skip connection

2. **Retrain** the model with the residual connection
3. Plot the training and validation losses and training and validation accuracy

In [ ]:
class CNNModelWithResidual(nn.Module):
    def __init__(self, num_classes=3):
        super(CNNModelWithResidual, self).__init__()
        
        # Conv Layer 1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        # Conv Layer 2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # Conv Layer 3
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        
        # Conv Layer 4 (receives skip from layer 2)
        # After concatenation: 64 + 64 = 128 channels
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        # Conv Layer 5
        self.conv5 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)
        
        # Activation and Pooling
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully Connected Layers
        self.fc1 = nn.Linear(512 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
    
    def forward(self, x):
        # Layer 1: 128x128 -> 64x64
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        
        # Layer 2: 64x64 -> 32x32
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        skip = x  # Save for residual connection
        
        # Layer 3: 32x32 -> 16x16
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        
        # Downsample skip to match size (32x32 -> 16x16)
        skip = self.pool(skip)
        
        # Concatenate skip connection
        x = torch.cat([x, skip], dim=1)  # 64 + 64 = 128 channels
        
        # Layer 4: 16x16 -> 8x8
        x = self.pool(self.relu(self.bn4(self.conv4(x))))
        
        # Layer 5: 8x8 -> 4x4
        x = self.pool(self.relu(self.bn5(self.conv5(x))))
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # Fully Connected
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        
        return x

# Test model
model_res = CNNModelWithResidual(num_classes=3)
print(model_res)

In [ ]:
# Retrain residual model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_res = CNNModelWithResidual(num_classes=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_res.parameters(), lr=0.001)

num_epochs = 10

train_losses_res = []
val_losses_res = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model_res, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model_res, test_loader, criterion, device)
    
    train_losses_res.append(train_loss)
    val_losses_res.append(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.2f}%, Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%")

In [ ]:
# Compare losses
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Original Train')
plt.plot(train_losses_res, label='Residual Train')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Comparison')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(val_losses, label='Original Val')
plt.plot(val_losses_res, label='Residual Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Validation Loss Comparison')
plt.legend()

plt.tight_layout()
plt.show()